# Final pipeline — robust normalization + rotation augmentation, CNN and GRU, ensembles

**Why this notebook.** Stage 4c showed that the old hand-size normalization (one projected bone) collapses in about 5 % of the frames and that a robust
scale (mean 3D length of four palm bones) gives the CNN **+4.0 points** (5 of 5 folds). All earlier results for the GRU and the ensembles used the old normalization.
This notebook makes `robust3d` + rotation augmentation the default pipeline and re-runs the two best architectures with it, with a protocol that also answers an open question.

**Bars to beat** (5-fold CV, 300 signs, chance 0.33 %): Stage 3 CNN 63.6 %, Stage 4b CNN + rotation 66.0 %, Stage 4b GRU + rotation 62.8 %,
Stage 4b ensemble of CNN + GRU **67.7 ± 3.7 %**, Stage 4c CNN + rotation + robust3d **68.6 ± 3.9 %**.

**What the notebook does**
1. Loads the `robust3d` sequences (built in Stage 4c; rebuilt here if missing) and repeats the unit tests and the memorization check.
2. **CNN:** 3 seeds on the single split (with a seed ensemble), 5-fold CV with **two seeds per fold**, and the signer-leakage test.
3. **GRU** (flag `RUN_GRU`): 3 seeds, 5-fold CV with one seed, leakage test.
4. **Ensembles in cross-validation:** two CNN seeds (plain averaging), CNN + GRU (different architectures), and all three. Comparing the first two answers the question left open in Stage 4b:
   does the ensemble gain come from *architectural diversity* or just from averaging two models?
5. Builds the consolidated table of all stages and **saves the cross-validated predictions** (`results/final_cv_logits.npz`) so that the error analysis can be done without retraining.

**How to run:** a new Colab notebook with a **GPU runtime**; run the cells in order from Cell 2. Needs in Drive: `splits.csv`, `landmarks_raw.npz`, `frames_tracked_v2.npy`
and, for the paired comparisons, the `results/stage*_results.csv` files and `stage4b_recipe.json`.
Rough time (an estimate, not measured): CNN part about 25 minutes, GRU part about 40 minutes. Results are saved after every phase. If a cell fails, send me the last lines of the error.

In [ ]:
# [setup] Cell 2 — configuration
from pathlib import Path
import json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/slovo_project')
except ImportError:                      # running outside Colab
    BASE = Path('./slovo_project')

PROC, FIG, RES = BASE / 'processed', BASE / 'figures', BASE / 'results'
for d in (FIG, RES):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
FIXED_T = 64
N_FOLDS = 5
FINAL_SEEDS = [0, 1, 2]
RUN_GRU = True                # the GRU part (about 40 minutes)
warnings.filterwarnings('ignore')
print('torch', torch.__version__, '| device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'no GPU')

In [ ]:
# Cell 3 — load the Stage 0 outputs (same files and same split as Stages 1 to 4)
splits = pd.read_csv(PROC / 'splits.csv')
z = np.load(PROC / 'landmarks_raw.npz', allow_pickle=False)
ids, lengths = z['ids'], z['lengths']
offsets = np.concatenate([[0], np.cumsum(lengths)])
id2idx = {c: i for i, c in enumerate(ids)}
frames = np.load(PROC / 'frames_tracked_v2.npy', mmap_mode='r')
assert len(frames) == offsets[-1], 'frames_tracked_v2.npy does not match landmarks_raw.npz: re-run Stage 0 cells 10 and 11b.'

meta = splits[splits['in_subset']].reset_index(drop=True)
y_all = meta['label_id_subset'].astype(int).to_numpy()
N_CLASSES = int(y_all.max()) + 1
assert len(np.unique(y_all)) == N_CLASSES

tr = (meta['split'] == 'train').to_numpy(); va = (meta['split'] == 'val').to_numpy(); te = (meta['split'] == 'test').to_numpy()
folds = meta['fold'].to_numpy()
print(f'{len(meta)} clips | {N_CLASSES} signs | chance {1 / N_CLASSES:.2%} | train {tr.sum()} val {va.sum()} test {te.sum()}')

In [ ]:
# [variants] Cell 4 — the robust3d sequences: 64 frames x 132 numbers per clip (built in Stage 4c; rebuilt here if missing)
from tqdm.auto import tqdm
WRIST, PALM = 0, [5, 9, 13, 17]          # wrist and the four finger bases (index, middle, ring, pinky)

def normalize_hands_robust(x, aspect):
    """x: (..., 21, 3) raw landmarks -> wrist-centred hand shape divided by the mean 3D length of four palm bones.
    x and z are put in image-height units with the aspect ratio. NaN where the hand is missing."""
    x = x.astype(np.float32).copy()
    x[..., 0] *= aspect
    x[..., 2] *= aspect
    centred = x - x[..., WRIST:WRIST + 1, :]
    scale = np.linalg.norm(centred[..., PALM, :], axis=-1).mean(-1)
    scale = np.where(scale < 1e-6, np.nan, scale)
    return centred / scale[..., None, None]

def clip_to_sequence(clip, aspect, T=FIXED_T):
    """(L, 2, 21, 3) -> (T, 132): 126 joint values (hand 0 then hand 1), 4 wrist-position values, 2 visibility flags."""
    L = len(clip)
    idx = np.linspace(0, L - 1, T).round().astype(int)
    hands = normalize_hands_robust(clip, aspect)[idx]
    wrist = clip[:, :, WRIST, :2].astype(np.float32).copy()
    wrist[..., 0] *= aspect
    wrist = wrist[idx]
    present = ~np.isnan(hands).all(axis=(2, 3))
    hands = np.where(np.isnan(hands), 0.0, hands)
    wrist = np.where(present[:, :, None], wrist, 0.0)
    return np.concatenate([hands.reshape(T, 126), wrist.reshape(T, 4), present.astype(np.float32)], axis=1).astype(np.float32)

_rng = np.random.default_rng(0)
_clip = _rng.normal(size=(40, 2, 21, 3)).astype(np.float32); _clip[10:15, 1] = np.nan
_s = clip_to_sequence(_clip, 1.0)
_m = clip_to_sequence(_clip * 2.0 + np.array([5.0, -3.0, 4.0], np.float32), 1.0)
assert _s.shape == (FIXED_T, 132) and np.isfinite(_s).all()
assert np.allclose(_s[:, :126], _m[:, :126], atol=1e-4), 'joint channels must not depend on hand position or size'
print('OK: robust3d unit tests passed')

SEQF = PROC / f'sequences_subset_T{FIXED_T}_robust3d.npz'
meta_ids = meta['attachment_id'].to_numpy().astype(str)
SEQ = None
if SEQF.exists():
    zs = np.load(SEQF, allow_pickle=False)
    if zs['ids'].shape == meta_ids.shape and (zs['ids'] == meta_ids).all():
        SEQ = zs['S'].astype(np.float32)
        print('Loaded cached sequences:', SEQF.name)
if SEQ is None:
    SEQ = np.zeros((len(meta), FIXED_T, 132), np.float32)
    for r, (cid, asp) in enumerate(tqdm(zip(meta['attachment_id'], meta['aspect']), total=len(meta))):
        i = id2idx[cid]
        SEQ[r] = clip_to_sequence(np.asarray(frames[offsets[i]:offsets[i + 1]], dtype=np.float32), float(asp))
    np.savez_compressed(SEQF, S=SEQ.astype(np.float16), ids=meta_ids)
    print('Saved', SEQF.name)
assert SEQ.shape == (len(meta), FIXED_T, 132) and np.isfinite(SEQ).all()
vis = SEQ[tr][:, :, 130:132].astype(bool)
vals = np.abs(SEQ[tr][:, :, :126].reshape(int(tr.sum()), FIXED_T, 2, 63)[vis])
print(f'normalized values (training clips): median {np.median(vals):.2f} | 99th pct {np.percentile(vals, 99):.2f} | 99.9th pct {np.percentile(vals, 99.9):.2f} | max {vals.max():.2f}')

In [ ]:
# [aug] Cell 5 — the augmentations (numpy, applied to RAW normalized hand shapes) and their unit tests
def seq_mask(S):
    """(N, T, 132) mask: 1 where the channel's hand is visible (or the channel is a visibility flag)."""
    p = S[:, :, 130:132]
    return np.concatenate([np.repeat(p[:, :, :1], 63, 2), np.repeat(p[:, :, 1:], 63, 2),
                           np.repeat(p[:, :, :1], 2, 2), np.repeat(p[:, :, 1:], 2, 2), np.ones_like(p)], axis=2)

def seq_stats(S_tr):
    m = seq_mask(S_tr)
    cnt = np.maximum(m.sum((0, 1)), 1.0)
    mean = (S_tr * m).sum((0, 1)) / cnt
    std = np.sqrt((((S_tr - mean) ** 2) * m).sum((0, 1)) / cnt)
    std = np.where(std < 1e-3, 1.0, std)
    mean[130:], std[130:] = 0.0, 1.0
    return mean.astype(np.float32), std.astype(np.float32)

def apply_seq_norm(S, mean, std):
    return (((S - mean) / std) * seq_mask(S)).astype(np.float32)           # missing hands stay exactly 0

BASE_MAG = {'rotation': 15.0, 'scale': 0.15, 'timewarp': 0.20, 'noise': 0.03,
            'framehold': 0.10, 'jointdrop': 0.05, 'wristjitter': 0.05}
ORDER = ['timewarp', 'framehold', 'rotation', 'scale', 'noise', 'jointdrop', 'wristjitter']
P_APPLY = 0.5                                                              # each augmentation is applied to a clip with this probability

def _joints(x):
    return x[:, :, :126].reshape(len(x), x.shape[1], 2, 21, 3)

def _put(x, j):
    out = x.copy()
    out[:, :, :126] = j.reshape(len(x), x.shape[1], 126)
    return out

def _on(rng, B):
    return (rng.random(B) < P_APPLY).astype(np.float32)

def aug_rotation(x, rng, mag):                                             # mag = maximum angle in degrees
    B = len(x)
    th = np.deg2rad(rng.uniform(-mag, mag, B)) * _on(rng, B)
    c = np.cos(th).reshape(B, 1, 1, 1).astype(np.float32); s = np.sin(th).reshape(B, 1, 1, 1).astype(np.float32)
    j = _joints(x).copy(); xx, yy = j[..., 0].copy(), j[..., 1].copy()
    j[..., 0] = c * xx - s * yy
    j[..., 1] = s * xx + c * yy
    return _put(x, j)

def aug_scale(x, rng, mag):                                                # mag = maximum relative change
    B = len(x)
    f = (1 + rng.uniform(-mag, mag, B) * _on(rng, B)).astype(np.float32).reshape(B, 1, 1, 1, 1)
    return _put(x, _joints(x) * f)

def aug_timewarp(x, rng, mag):                                             # mag = maximum relative speed change (shift = mag * 20 frames)
    B, T = x.shape[:2]
    on = _on(rng, B)
    r = 1 + rng.uniform(-mag, mag, B) * on
    sh = rng.uniform(-mag, mag, B) * 20 * on
    t, c = np.arange(T)[None, :], (T - 1) / 2
    idx = np.clip(np.rint(c + (t - c) * r[:, None] + sh[:, None]), 0, T - 1).astype(int)
    return np.take_along_axis(x, idx[:, :, None], axis=1)

def aug_framehold(x, rng, mag):                                            # mag = probability that a frame repeats the previous one
    B, T = x.shape[:2]
    drop = (rng.random((B, T)) < min(mag, 0.5)) & (_on(rng, B)[:, None] > 0)
    drop[:, 0] = False
    idx = np.maximum.accumulate(np.where(drop, 0, np.arange(T)[None, :]), axis=1)    # last kept frame
    return np.take_along_axis(x, idx[:, :, None], axis=1)

def aug_noise(x, rng, mag):                                                # mag = standard deviation, visible hands only
    B, T = x.shape[:2]
    vis = x[:, :, 130:132].reshape(B, T, 2, 1, 1)
    n = rng.normal(0, mag, (B, T, 2, 21, 3)).astype(np.float32) * vis * _on(rng, B).reshape(B, 1, 1, 1, 1)
    return _put(x, _joints(x) + n)

def aug_jointdrop(x, rng, mag):                                            # mag = probability that a joint is set to 0
    B, T = x.shape[:2]
    m = (rng.random((B, T, 2, 21, 1)) < min(mag, 0.5)) & (_on(rng, B).reshape(B, 1, 1, 1, 1) > 0)
    return _put(x, np.where(m, 0.0, _joints(x)).astype(np.float32))

def aug_wristjitter(x, rng, mag):                                          # mag = standard deviation of a constant shift, visible hands only
    B, T = x.shape[:2]
    vis = x[:, :, 130:132].reshape(B, T, 2, 1)
    off = rng.normal(0, mag, (B, 1, 2, 2)).astype(np.float32) * _on(rng, B).reshape(B, 1, 1, 1)
    out = x.copy()
    out[:, :, 126:130] = (x[:, :, 126:130].reshape(B, T, 2, 2) + off * vis).reshape(B, T, 4)
    return out

AUG_FUNCS = {'rotation': aug_rotation, 'scale': aug_scale, 'timewarp': aug_timewarp, 'noise': aug_noise,
             'framehold': aug_framehold, 'jointdrop': aug_jointdrop, 'wristjitter': aug_wristjitter}

def augment(x, names, rng, strength=1.0):
    """Apply the named augmentations (always in the same order) to a batch x of RAW sequences (B, T, 132)."""
    for name in ORDER:
        if name in names:
            x = AUG_FUNCS[name](x, rng, BASE_MAG[name] * strength)
    return x

# --- unit tests -------------------------------------------------------------------------------------------------------
_rng = np.random.default_rng(0)
_x = SEQ[tr][:32].copy()

def _consistent(o):
    """Wherever a hand's visibility flag is 0, that hand's joints and wrist position must be exactly 0."""
    B, T = o.shape[:2]
    inv = 1.0 - o[:, :, 130:132]
    return (np.abs(o[:, :, :126].reshape(B, T, 2, 63)) * inv[..., None]).max() == 0 and \
           (np.abs(o[:, :, 126:130].reshape(B, T, 2, 2)) * inv[..., None]).max() == 0

assert _consistent(_x), 'the input data itself is inconsistent'
for name in ORDER:
    out = augment(_x, (name,), _rng)
    assert out.shape == _x.shape and out.dtype == np.float32 and np.isfinite(out).all(), name
    assert _consistent(out), f'{name}: a hand that is flagged missing has non-zero values'
    if name not in ('timewarp', 'framehold'):                                  # these two only reorder frames in time
        assert np.array_equal(out[:, :, 130:132], _x[:, :, 130:132]), f'{name} changed the visibility flags'
    assert not np.array_equal(out, _x) or name in ('timewarp', 'framehold'), f'{name} did nothing'
    zero = AUG_FUNCS[name](_x, _rng, 0.0)                                      # magnitude 0 = identity
    assert np.allclose(zero, _x, atol=1e-6), f'{name} with magnitude 0 is not the identity'
# rotation keeps distances; framehold repeats the previous frame; timewarp at speed 1 and shift 0 is the identity
P_APPLY_BACKUP, P_APPLY = P_APPLY, 1.0
r = aug_rotation(_x, _rng, 30.0)
assert np.allclose(np.linalg.norm(_joints(r)[..., :2], axis=-1), np.linalg.norm(_joints(_x)[..., :2], axis=-1), atol=1e-4), 'rotation changed distances'
h = aug_framehold(_x, _rng, 0.3)
for b in range(len(h)):
    for t in range(1, h.shape[1]):
        assert np.array_equal(h[b, t], _x[b, t]) or np.array_equal(h[b, t], h[b, t - 1]), 'framehold produced a frame that is neither new nor repeated'
P_APPLY = P_APPLY_BACKUP
print('OK: augmentation unit tests passed for', ORDER)

In [ ]:
# [torch] Cell 6 — the two networks (CNN and GRU, both take (batch, time, channels)) and the training loop with augmentation
import torch.nn as nn
import torch.nn.functional as F

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class CNN1D(nn.Module):
    """Stage 3 network: blocks of [Conv1d -> BatchNorm -> ReLU -> MaxPool(2) -> Dropout], global average + max pooling, Linear."""
    def __init__(self, in_ch, n_classes, channels=(128, 128, 256), kernel=9, dropout=0.5, block_dropout=0.1):
        super().__init__()
        blocks, c_in = [], in_ch
        for c_out in channels:
            blocks += [nn.Conv1d(c_in, c_out, kernel, padding=kernel // 2), nn.BatchNorm1d(c_out), nn.ReLU(),
                       nn.MaxPool1d(2), nn.Dropout(block_dropout)]
            c_in = c_out
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * c_in, n_classes))

    def forward(self, x):                                         # x: (batch, time, channels)
        h = self.features(x.transpose(1, 2))
        h = torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1)
        return self.head(h)

class GRUNet(nn.Module):
    """Stage 4 network: optional conv front-end, (Bi)GRU, pooling over time, Linear. Input (batch, time, channels)."""
    def __init__(self, in_ch, n_classes, hidden=256, layers=2, bidirectional=True, dropout=0.5, rnn_dropout=0.3,
                 pooling='meanmax', frontend='conv', conv_ch=128):
        super().__init__()
        if frontend == 'conv':
            self.frontend = nn.Sequential(nn.Conv1d(in_ch, conv_ch, 5, padding=2), nn.BatchNorm1d(conv_ch), nn.ReLU())
            rnn_in = conv_ch
        else:
            self.frontend, rnn_in = None, in_ch
        self.rnn = nn.GRU(rnn_in, hidden, num_layers=layers, batch_first=True, bidirectional=bidirectional,
                          dropout=rnn_dropout if layers > 1 else 0.0)
        d = hidden * (2 if bidirectional else 1)
        self.pooling, self.bidirectional = pooling, bidirectional
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * d if pooling == 'meanmax' else d, n_classes))

    def forward(self, x):
        if self.frontend is not None:
            x = self.frontend(x.transpose(1, 2)).transpose(1, 2)
        out, h_n = self.rnn(x)
        if self.pooling == 'meanmax':
            h = torch.cat([out.mean(dim=1), out.amax(dim=1)], dim=1)
        elif self.bidirectional:
            h = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            h = h_n[-1]
        return self.head(h)

def new_model(cfg, in_ch, seed):
    set_seed(seed)
    if cfg['kind'] == 'cnn':
        m = CNN1D(in_ch, N_CLASSES, cfg['channels'], cfg['kernel'], cfg['dropout'], cfg['block_dropout'])
    else:
        m = GRUNet(in_ch, N_CLASSES, cfg['hidden'], cfg['layers'], cfg['bidirectional'], cfg['dropout'], cfg['rnn_dropout'],
                   cfg['pooling'], cfg['frontend'], cfg['conv_ch'])
    return m.to(DEVICE)

@torch.no_grad()
def check_architecture(cfg, in_ch=132):
    """Dummy batch through the network: returns (output shape, number of parameters)."""
    m = new_model(cfg, in_ch, 0).eval()
    out = m(torch.zeros(2, FIXED_T, in_ch, device=DEVICE))
    return tuple(out.shape), sum(p.numel() for p in m.parameters())

@torch.no_grad()
def _eval(model, X, y):
    model.eval()
    logits = model(X)
    return F.cross_entropy(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

def fit(model, Xtr_raw, ytr, mean, std, Xva=None, yva=None, aug=(), strength=1.0, epochs=200, lr=1e-3, weight_decay=1e-2,
        batch_size=128, patience=25, label_smoothing=0.1, seed=0, fixed_epochs=None, grad_clip=0.0):
    """Mini-batch training as in Stages 2 to 4. Xtr_raw are the RAW training sequences (N, T, 132); when `aug` is not empty,
    every mini-batch is augmented on the fly (numpy) and then standardized with the TRAIN mean/std. Xva is already standardized."""
    t0 = time.time()
    n_epochs = fixed_epochs or epochs
    g = torch.Generator(); g.manual_seed(seed)                              # order of the mini-batches
    arng = np.random.default_rng(10_000 + seed)                             # randomness of the augmentation
    Xclean = torch.as_tensor(apply_seq_norm(Xtr_raw, mean, std), dtype=torch.float32, device=DEVICE)   # for the training-accuracy curve
    yt = torch.as_tensor(ytr, dtype=torch.long, device=DEVICE)
    use_val = (Xva is not None) and (fixed_epochs is None)
    if use_val:
        Xv = torch.as_tensor(Xva, dtype=torch.float32, device=DEVICE)
        yv = torch.as_tensor(yva, dtype=torch.long, device=DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_epoch, best_state, bad, n = -1.0, 0, None, 0, len(Xtr_raw)
    for epoch in range(1, n_epochs + 1):
        model.train()
        perm = torch.randperm(n, generator=g)                              # CPU tensor
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            if aug:
                xb = torch.as_tensor(apply_seq_norm(augment(Xtr_raw[idx.numpy()], aug, arng, strength), mean, std),
                                     dtype=torch.float32, device=DEVICE)
            else:
                xb = Xclean[idx.to(DEVICE)]
            yb = yt[idx.to(DEVICE)]
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
        tl, ta = _eval(model, Xclean, yt)
        hist['train_loss'].append(tl); hist['train_acc'].append(ta)
        if use_val:
            vl, vacc = _eval(model, Xv, yv)
            hist['val_loss'].append(vl); hist['val_acc'].append(vacc)
            if vacc > best_acc:
                best_acc, best_epoch, bad = vacc, epoch, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    break
    if use_val and best_state is not None:
        model.load_state_dict(best_state)
    else:
        best_epoch = n_epochs
    return hist, best_epoch, time.time() - t0

@torch.no_grad()
def predict_logits(model, X, batch_size=512):
    model.eval()
    out = []
    for i in range(0, len(X), batch_size):
        xb = torch.as_tensor(X[i:i + batch_size], dtype=torch.float32, device=DEVICE)
        out.append(model(xb).cpu().numpy())
    return np.concatenate(out)

print('networks and training loop defined')

In [ ]:
# [runner] Cell 7 — configurations, metrics and the experiment runner
from sklearn.metrics import f1_score

CNN_CFG = dict(kind='cnn', channels=(128, 128, 256), kernel=9, dropout=0.5, block_dropout=0.1, grad_clip=0.0)   # Stage 3 final
GRU_CFG = dict(kind='gru', hidden=256, layers=2, bidirectional=True, dropout=0.5, rnn_dropout=0.3, pooling='meanmax',
               frontend='conv', conv_ch=128, grad_clip=1.0)                                                      # Stage 4 final
TRAIN_CFG = dict(lr=1e-3, weight_decay=1e-2, batch_size=128, epochs=200, patience=25, label_smoothing=0.1)      # more room than before: augmentation converges slower

for nm, cfg in (('CNN', CNN_CFG), ('GRU', GRU_CFG)):
    shape, n_par = check_architecture(cfg)
    assert shape == (2, N_CLASSES), f'unexpected output shape {shape}'
    print(f'{nm}: output {shape} | {n_par:,} parameters')

def metrics_from_logits(logits, y):
    pred = logits.argmax(1)
    top5 = np.argsort(-logits, axis=1)[:, :5]
    return {'n': int(len(y)), 'top1': float((pred == y).mean()), 'top5': float(np.mean([yt in t for yt, t in zip(y, top5)])),
            'macro_f1': float(f1_score(y, pred, average='macro')), 'pred': pred, 'logits': logits.astype(np.float32)}

def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

MASKS = {'train': tr, 'val': va, 'test': te}
results = []
_cache = {}

def mask_key(masks):
    return tuple((k, hash(m.tobytes())) for k, m in masks.items())

def run_one(experiment, name, model_cfg, aug, strength, masks, seed, fixed_epochs=None, use_cache=True):
    """Train one model on masks['train'] (augmenting if `aug` is not empty), evaluate on every other mask."""
    ckey = (json.dumps(model_cfg, sort_keys=True, default=list), tuple(aug), float(strength), seed, mask_key(masks), fixed_epochs)
    if use_cache and ckey in _cache:
        return _cache[ckey]
    Xtr = SEQ[masks['train']]
    mean, std = seq_stats(Xtr)
    ev = {k: apply_seq_norm(SEQ[m], mean, std) for k, m in masks.items() if k != 'train'}
    model = new_model(model_cfg, 132, seed)
    hist, best_ep, fit_s = fit(model, Xtr, y_all[masks['train']], mean, std, ev.get('val'), y_all[masks['val']] if 'val' in masks else None,
                               aug=tuple(aug), strength=strength, seed=seed, fixed_epochs=fixed_epochs,
                               grad_clip=model_cfg['grad_clip'], **TRAIN_CFG)
    out = {'name': name, 'seed': seed, 'hist': hist, 'best_epoch': best_ep, 'fit_s': fit_s, 'metrics': {}}
    for sname, m in masks.items():
        if sname == 'train':
            continue
        r = metrics_from_logits(predict_logits(model, ev[sname]), y_all[m])
        out['metrics'][sname] = r
        results.append({'experiment': experiment, 'name': name, 'aug': '+'.join(aug) or 'none', 'strength': strength, 'seed': seed,
                        'set': sname, 'n': r['n'], 'top1': r['top1'], 'top5': r['top5'], 'macro_f1': r['macro_f1'],
                        'best_epoch': best_ep, 'fit_s': fit_s})
    del model
    if use_cache:
        _cache[ckey] = out
    return out

def val_of(outs):
    v = [o['metrics']['val']['top1'] for o in outs]
    return float(np.mean(v)), float(np.std(v))

def summary_line(o, sets=('val', 'test')):
    return ' | '.join(f'{s} top-1 {o["metrics"][s]["top1"]:.1%} top-5 {o["metrics"][s]["top5"]:.1%}' for s in sets if s in o['metrics']) \
           + f' | best epoch {o["best_epoch"]} | {o["fit_s"]:.0f}s'

def save_results(tag=''):
    pd.DataFrame(results).to_csv(RES / 'final_results.csv', index=False)

In [ ]:
# Cell 8 — the augmentation recipe chosen in Stage 4b (rotation, strength 1.0) and the sanity check
rf = RES / 'stage4b_recipe.json'
if rf.exists():
    rj = json.load(open(rf)); RECIPE, STRENGTH = tuple(rj['recipe']), float(rj['strength'])
else:
    RECIPE, STRENGTH = ('rotation',), 1.0
print('recipe:', RECIPE, 'strength', STRENGTH)

Xtr = SEQ[tr]; mean, std = seq_stats(Xtr)
small_i = np.random.default_rng(0).choice(len(Xtr), 64, replace=False)
cfg0 = {**CNN_CFG, 'dropout': 0.0, 'block_dropout': 0.0}
mdl = new_model(cfg0, 132, 0)
h, _, _ = fit(mdl, Xtr[small_i], y_all[tr][small_i], mean, std, aug=RECIPE, strength=0.5, fixed_epochs=200,
              lr=3e-3, weight_decay=0.0, batch_size=64, label_smoothing=0.0, seed=0)
print(f'train loss {h["train_loss"][0]:.2f} -> {h["train_loss"][-1]:.3f} | clean train accuracy {h["train_acc"][0]:.0%} -> {h["train_acc"][-1]:.0%}')
assert h['train_acc'][-1] > 0.8, 'The CNN cannot memorize 64 clips: something is wrong (send me this output).'
print('OK: the training pipeline works')
del mdl, Xtr

In [ ]:
# Cell 9 — CNN, 3 seeds on the single split, and the seed ensemble
def softmax_ensemble(outs, sname):
    P = np.mean([softmax(o['metrics'][sname]['logits']) for o in outs], axis=0)
    return metrics_from_logits(np.log(P + 1e-9), y_all[MASKS[sname]])

fin = {}
t0 = time.time()
fin['CNN'] = [run_one('H1 CNN seeds', 'CNN robust3d + rotation', CNN_CFG, RECIPE, STRENGTH, MASKS, s, use_cache=False) for s in FINAL_SEEDS]
v = [o['metrics']['val']['top1'] for o in fin['CNN']]; t = [o['metrics']['test']['top1'] for o in fin['CNN']]; t5 = [o['metrics']['test']['top5'] for o in fin['CNN']]
e = softmax_ensemble(fin['CNN'], 'test')
print(f'CNN robust3d + rotation: val {np.mean(v):.1%} +/- {np.std(v):.1%} | test {np.mean(t):.1%} +/- {np.std(t):.1%} | test top-5 {np.mean(t5):.1%} | per-seed test: ' + ', '.join(f'{x:.1%}' for x in t))
print(f'ensemble of the 3 seeds: test top-1 {e["top1"]:.1%} top-5 {e["top5"]:.1%}   ({(time.time() - t0) / 60:.0f} min)')
print('(Stage 4c, same configuration: val 68.6 +/- 1.3, test 71.5 +/- 0.1)')
save_results()

In [ ]:
# Cell 10 — CNN, 5-fold cross-validation with TWO seeds per fold (same folds as Stages 1 to 4c) and the paired comparisons
def cv_run(tag, model_cfg, seed, experiment):
    acc, logits, ys, fold_ids = [], [], [], []
    for k in range(N_FOLDS):
        te_k = folds == k; va_k = folds == (k + 1) % N_FOLDS; tr_k = ~(te_k | va_k)
        o = run_one(experiment, tag, model_cfg, RECIPE, STRENGTH, {'train': tr_k, 'val': va_k, 'test': te_k}, seed, use_cache=False)
        results[-2]['set'], results[-1]['set'] = f'fold {k} val', f'fold {k}'
        acc.append(o['metrics']['test']['top1']); logits.append(o['metrics']['test']['logits']); ys.append(y_all[te_k]); fold_ids.append(meta_ids[te_k])
        print(f'  {tag} seed {seed} fold {k}: test clips {te_k.sum()} | top-1 {acc[-1]:.1%} | best epoch {o["best_epoch"]} | {o["fit_s"]:.0f}s')
    acc = np.array(acc)
    print(f'{tag} seed {seed}: top-1 mean {acc.mean():.1%} +/- {acc.std():.1%} over folds | pooled {np.mean(np.concatenate([l.argmax(1) for l in logits]) == np.concatenate(ys)):.1%}\n')
    save_results()
    return {'acc': acc, 'logits': logits, 'y': ys, 'ids': fold_ids}

CV = {}
CV['CNN s0'] = cv_run('CNN robust3d + rotation', CNN_CFG, 0, 'H2 CNN 5-fold CV')
CV['CNN s1'] = cv_run('CNN robust3d + rotation', CNN_CFG, 1, 'H2 CNN 5-fold CV')

def cv_from_csv(path, experiment, name):
    if not path.exists():
        return None
    d = pd.read_csv(path)
    col = 'name' if 'name' in d else 'model'
    d = d[(d.experiment == experiment) & (d[col] == name) & d.set.str.fullmatch(r'fold \d')]
    v = d.sort_values('set').top1.to_numpy()
    return v if len(v) == N_FOLDS else None

REFS = {'Stage 1 linear SVM': cv_from_csv(RES / 'stage1_results.csv', '5-fold CV', 'linear SVM'),
        'Stage 2 MLP on features': cv_from_csv(RES / 'stage2_results.csv', 'E5 5-fold CV', 'MLP on features'),
        'Stage 3 CNN (no aug)': cv_from_csv(RES / 'stage3_results.csv', 'E4 5-fold CV', 'CNN tuned'),
        'Stage 4 GRU (no aug)': cv_from_csv(RES / 'stage4_results.csv', 'E4 5-fold CV', 'GRU tuned'),
        'Stage 4b CNN + rotation': cv_from_csv(RES / 'stage4b_results.csv', 'F2 CNN 5-fold CV', 'CNN + aug'),
        'Stage 4b GRU + rotation': cv_from_csv(RES / 'stage4b_results.csv', 'G2 GRU 5-fold CV', 'GRU + aug'),
        'Stage 4c CNN robust3d': cv_from_csv(RES / 'stage4c_results.csv', 'V2 variants 5-fold CV', 'CNN robust3d')}

def compare(tag, acc, others):
    for nm, vv in others.items():
        if vv is None:
            continue
        diff = acc - vv
        print(f'  {tag} vs {nm:26s} {vv.mean():.1%} +/- {vv.std():.1%} | better in {(diff > 0).sum()} of {N_FOLDS} folds | mean difference {diff.mean():+.1%} '
              f'(std error {diff.std(ddof=1) / np.sqrt(N_FOLDS):.1%}; per fold: ' + ', '.join(f'{x:+.1%}' for x in diff) + ')')

both = (CV['CNN s0']['acc'] + CV['CNN s1']['acc']) / 2
print(f'CNN, mean of the two seeds per fold: {both.mean():.1%} +/- {both.std():.1%}   (the difference between the two seeds in a fold: ' +
      ', '.join(f'{x:+.1%}' for x in CV["CNN s1"]["acc"] - CV["CNN s0"]["acc"]) + ')')
print('\nPaired comparison of the CNN (seed 0):')
compare('CNN s0', CV['CNN s0']['acc'], REFS)

In [ ]:
# Cell 11 — CNN, signer-leakage re-test (official split, seen vs unseen signers)
off_train = meta['official_train'].to_numpy(bool)
unseen = meta['official_test_unseen_signer'].to_numpy(bool)
seen = (~off_train) & (~unseen)
print(f'official train {off_train.sum()} | official test: {seen.sum()} clips from seen signers, {unseen.sum()} from unseen signers')

def leakage(tag, model_cfg, epochs, experiment):
    runs = [run_one(experiment, tag, model_cfg, RECIPE, STRENGTH, {'train': off_train, 'seen': seen, 'unseen': unseen}, s,
                    fixed_epochs=epochs, use_cache=False) for s in FINAL_SEEDS]
    a = [o['metrics']['seen']['top1'] for o in runs]; b_ = [o['metrics']['unseen']['top1'] for o in runs]
    print(f'{tag:26s} (fixed {epochs} epochs) seen {np.mean(a):.1%} +/- {np.std(a):.1%} | unseen {np.mean(b_):.1%} +/- {np.std(b_):.1%} | gap {np.mean(a) - np.mean(b_):+.1%}')
    save_results()
    return np.mean(a) - np.mean(b_), np.mean(b_)

LEAK = {}
LEAK['CNN'] = leakage('CNN robust3d + rotation', CNN_CFG, int(np.median([o['best_epoch'] for o in fin['CNN']])), 'H3 CNN leakage')
print('(earlier CNN gaps: Stage 3 +4.4, Stage 4b without/with rotation +4.8/+5.9; the standard error of a gap is about 2.7 points)')

In [ ]:
# Cell 12 — GRU with robust3d + rotation: 3 seeds, 5-fold CV (one seed), leakage
if RUN_GRU:
    t_gru = time.time()
    fin['GRU'] = [run_one('I1 GRU seeds', 'GRU robust3d + rotation', GRU_CFG, RECIPE, STRENGTH, MASKS, s, use_cache=False) for s in FINAL_SEEDS]
    v = [o['metrics']['val']['top1'] for o in fin['GRU']]; t = [o['metrics']['test']['top1'] for o in fin['GRU']]
    e = softmax_ensemble(fin['GRU'], 'test')
    print(f'GRU robust3d + rotation: val {np.mean(v):.1%} +/- {np.std(v):.1%} | test {np.mean(t):.1%} +/- {np.std(t):.1%} | ensemble of 3: test top-1 {e["top1"]:.1%} top-5 {e["top5"]:.1%}'
          f'   (Stage 4b, old normalization: val 62.4, test 65.7 +/- 1.5)')
    save_results()
    CV['GRU s0'] = cv_run('GRU robust3d + rotation', GRU_CFG, 0, 'I2 GRU 5-fold CV')
    print('Paired comparison of the GRU:')
    compare('GRU s0', CV['GRU s0']['acc'], {**REFS, 'CNN robust3d (this notebook)': CV['CNN s0']['acc']})
    LEAK['GRU'] = leakage('GRU robust3d + rotation', GRU_CFG, int(np.median([o['best_epoch'] for o in fin['GRU']])), 'I3 GRU leakage')
    print('(Stage 4b GRU with rotation, old normalization: seen 56.6%, unseen 52.7%, gap +3.9)')
    print(f'GRU part took {(time.time() - t_gru) / 60:.0f} min')
else:
    print('RUN_GRU is False: skipped')

In [ ]:
# Cell 13 — ensembles: seeds on the single split, then the cross-validated ensembles that separate diversity from averaging
print('single split (test fold), ensemble = average of the predicted probabilities:')
for combo in [('CNN',), ('GRU',), ('CNN', 'GRU')]:
    if not all(c in fin for c in combo):
        continue
    outs = [o for c in combo for o in fin[c]]
    for sname in ('val', 'test'):
        e = softmax_ensemble(outs, sname)
        single = np.mean([o['metrics'][sname]['top1'] for o in outs])
        print(f'  {" + ".join(combo):10s} {len(outs)} models {sname:4s} top-1 {e["top1"]:.1%} top-5 {e["top5"]:.1%} | average single model {single:.1%} | gain {e["top1"] - single:+.1%}')

def cv_ensemble(keys):
    acc = []
    for k in range(N_FOLDS):
        P = np.mean([softmax(CV[key]['logits'][k]) for key in keys], axis=0)
        acc.append(float((P.argmax(1) == CV[keys[0]]['y'][k]).mean()))
    return np.array(acc)

print('\n5-fold cross-validated ensembles (same test clips in every fold):')
combos = {'two CNN seeds (averaging only)': ('CNN s0', 'CNN s1')}
if 'GRU s0' in CV:
    combos['CNN + GRU (different architectures)'] = ('CNN s0', 'GRU s0')
    combos['two CNN seeds + GRU'] = ('CNN s0', 'CNN s1', 'GRU s0')
ENS = {}
for nm, keys in combos.items():
    ENS[nm] = cv_ensemble(keys)
    print(f'  {nm:38s} {ENS[nm].mean():.1%} +/- {ENS[nm].std():.1%} | folds ' + ', '.join(f'{x:.1%}' for x in ENS[nm]))
print('\nPaired comparisons:')
compare('two CNN seeds', ENS['two CNN seeds (averaging only)'], {'single CNN (seed 0)': CV['CNN s0']['acc']})
if 'CNN + GRU (different architectures)' in ENS:
    compare('CNN + GRU', ENS['CNN + GRU (different architectures)'], {'single CNN (seed 0)': CV['CNN s0']['acc'], 'two CNN seeds': ENS['two CNN seeds (averaging only)']})
    compare('CNN x2 + GRU', ENS['two CNN seeds + GRU'], {'two CNN seeds': ENS['two CNN seeds (averaging only)'], 'CNN + GRU': ENS['CNN + GRU (different architectures)']})
print('(Stage 4b, old normalization: ensemble of CNN + GRU 67.7% +/- 3.7%)')

In [ ]:
# Cell 14 — the consolidated table, the figure, and the saved predictions
rows = []
def add(label, acc, group):
    if acc is not None:
        rows.append({'model': label, 'cv_mean': float(np.mean(acc)), 'cv_std': float(np.std(acc)), 'group': group})
for nm, vv in REFS.items():
    add(nm, vv, 'robust3d (Stage 4c)' if 'robust3d' in nm else 'earlier stages (old normalization)')
add('CNN robust3d + rotation (seed 0)', CV['CNN s0']['acc'], 'this notebook (robust3d)')
add('CNN robust3d + rotation (mean of 2 seeds per fold)', (CV['CNN s0']['acc'] + CV['CNN s1']['acc']) / 2, 'this notebook (robust3d)')
if 'GRU s0' in CV:
    add('GRU robust3d + rotation', CV['GRU s0']['acc'], 'this notebook (robust3d)')
for nm, a in ENS.items():
    add('Ensemble: ' + nm, a, 'this notebook (robust3d)')
tab = pd.DataFrame(rows)
tab.to_csv(RES / 'final_table.csv', index=False)
print(tab.assign(cv=lambda d: (d.cv_mean * 100).round(1).astype(str) + ' +/- ' + (d.cv_std * 100).round(1).astype(str))[['model', 'cv', 'group']].to_string(index=False))

plt.figure(figsize=(11, 4.6))
colors = ['#999' if g.startswith('earlier') else '#4C78A8' for g in tab.group]            # grey = old normalization, blue = robust3d
plt.bar(range(len(tab)), tab.cv_mean, yerr=tab.cv_std, capsize=3, color=colors)
plt.xticks(range(len(tab)), tab.model, rotation=40, ha='right', fontsize=8); plt.ylabel('top-1 (5-fold CV, mean +/- std)')
plt.axhline(1 / N_CLASSES, color='k', ls='--', lw=1); plt.title('All stages: old normalization (grey) vs the final pipeline (blue)')
plt.tight_layout(); plt.savefig(FIG / 'final_results.png', dpi=150); plt.show()

# save the cross-validated predictions so that the error analysis needs no retraining
arrays = {}
for key, d in CV.items():
    for k in range(N_FOLDS):
        arrays[f'{key.replace(" ", "_")}_fold{k}_logits'] = d['logits'][k].astype(np.float16)
for k in range(N_FOLDS):
    arrays[f'fold{k}_ids'] = CV['CNN s0']['ids'][k]; arrays[f'fold{k}_y'] = CV['CNN s0']['y'][k]
np.savez_compressed(RES / 'final_cv_logits.npz', **arrays)
pd.DataFrame(results).to_csv(RES / 'final_results.csv', index=False)
print('Saved', RES / 'final_table.csv', RES / 'final_results.csv', RES / 'final_cv_logits.npz', 'and', FIG / 'final_results.png')

## Final pipeline checklist

- [ ] Cell 4: `OK: robust3d unit tests passed` and the normalized-value statistics (median about 0.48, 99.9th percentile about 2.2)
- [ ] Cell 5: `OK: augmentation unit tests passed`; Cell 8: `OK: the training pipeline works`
- [ ] Cell 9: CNN 3 seeds and the seed ensemble
- [ ] Cell 10: CNN 5-fold CV (two seeds) and the paired comparisons
- [ ] Cell 11: CNN leakage
- [ ] Cell 12: GRU (if `RUN_GRU = True`)
- [ ] Cell 13: the cross-validated ensembles (two CNN seeds vs CNN + GRU)
- [ ] `results/final_table.csv`, `results/final_cv_logits.npz` and `figures/final_results.png` exist

**Send back:** the printed outputs of Cells 4, 5 and 8 to 14 and the figure `final_results.png`.
If a cell fails, send me the last lines of the error.